In [1]:
# Install required packages
!pip -q install sentence-transformers faiss-cpu pandas numpy


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from IPython.display import display

print("Libraries loaded successfully.")

c:\Users\KRUSHIL\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully.


In [2]:
# File paths
chunks_path = "./02_rmit_assessment_support_chunks.csv"
test_path = "./test_prompts.json"

# Load chunk metadata
chunks_df = pd.read_csv(chunks_path)

# Load test prompts
with open(test_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_df = pd.DataFrame(test_data["tests"])

print("Chunks:", len(chunks_df))
print("Test prompts:", len(test_df))

display(test_df)

Chunks: 71
Test prompts: 12


,prompt,chunks
0,I've got a timed quiz that I won't be able to ...,[EXT-01_1]
1,"I've recently lost my job, is this a valid rea...",[SC-01_7]
2,So my hard drive with my assignment stopped wo...,[SC-01_16]
3,I've just booked my first appointment with ELS...,[EAA-02_29]
4,I emailed my course coordinator three days ago...,[EAA-02_38]
5,Having a hard time reading through journal art...,[EAA-02_46]
6,My mate got flagged for plagiarism because he ...,[POL-01_70]
7,Who actually sets RMIT's official academic cal...,[POL-01_67]
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]"
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]"


In [3]:
# Load the same embedding model used in the RAG pipeline
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")
print("Model: all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2446.14it/s]


Embedding model loaded successfully.
Model: all-MiniLM-L6-v2


In [4]:
# Recreate the Iteration 2 embedding text
chunks_df["embedding_text"] = (
    chunks_df["category"].fillna("") + ". " +
    chunks_df["section"].fillna("") + ". " +
    chunks_df["content"].fillna("")
)

# Generate embeddings
embeddings = embedding_model.encode(
    chunks_df["embedding_text"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

# Normalise for cosine similarity using inner product
faiss.normalize_L2(embeddings)

# Create FAISS index
dimension = embeddings.shape[1]
improved_index = faiss.IndexFlatIP(dimension)
improved_index.add(embeddings)

print("FAISS index created successfully.")
print("Vectors:", improved_index.ntotal)
print("Embedding dimension:", dimension)

Batches: 100%|██████████| 3/3 [00:01<00:00,  1.68it/s]

FAISS index created successfully.
Vectors: 71
Embedding dimension: 384


In [5]:
def evaluate_test_prompts(test_df, search_index, top_k=5):
    results = []

    for _, test in test_df.iterrows():

        question = test["prompt"]
        expected_chunks = test["chunks"]

        # Embed the test question
        query_embedding = embedding_model.encode(
            [question],
            convert_to_numpy=True
        ).astype("float32")

        # Normalise query for cosine similarity
        faiss.normalize_L2(query_embedding)

        # Retrieve top-k chunks
        scores, indices = search_index.search(
            query_embedding,
            top_k
        )

        retrieved_chunks = []
        retrieved_scores = []

        for idx, score in zip(indices[0], scores[0]):
            row = chunks_df.iloc[idx]
            retrieved_chunks.append(row["chunk_id"])
            retrieved_scores.append(round(float(score), 4))

        top1_chunks = retrieved_chunks[:1]
        top3_chunks = retrieved_chunks[:3]
        top5_chunks = retrieved_chunks[:5]

        # Hit = at least one expected chunk appears in the retrieved list
        hit_top1 = any(chunk in top1_chunks for chunk in expected_chunks)
        hit_top3 = any(chunk in top3_chunks for chunk in expected_chunks)
        hit_top5 = any(chunk in top5_chunks for chunk in expected_chunks)

        # Record rank of the first expected chunk, if retrieved
        expected_ranks = [
            retrieved_chunks.index(chunk) + 1
            for chunk in expected_chunks
            if chunk in retrieved_chunks
        ]

        results.append({
            "prompt": question,
            "expected_chunks": expected_chunks,
            "top1_chunk": top1_chunks[0],
            "top3_chunks": top3_chunks,
            "top5_chunks": top5_chunks,
            "top5_scores": retrieved_scores,
            "expected_rank_in_top5": min(expected_ranks) if expected_ranks else None,
            "top1_hit": hit_top1,
            "top3_hit": hit_top3,
            "top5_hit": hit_top5
        })

    return pd.DataFrame(results)

In [6]:
# Run evaluation
evaluation_df = evaluate_test_prompts(
    test_df,
    improved_index,
    top_k=5
)

display(evaluation_df)

,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,top5_scores,expected_rank_in_top5,top1_hit,top3_hit,top5_hit
0,I've got a timed quiz that I won't be able to ...,[EXT-01_1],EXT-01_1,"[EXT-01_1, EXT-01_0, POL-01_62]","[EXT-01_1, EXT-01_0, POL-01_62, EXT-01_2, EAA-...","[0.6777, 0.6126, 0.4892, 0.4775, 0.4761]",1.0,True,True,True
1,"I've recently lost my job, is this a valid rea...",[SC-01_7],SC-01_4,"[SC-01_4, SC-01_7, SC-01_13]","[SC-01_4, SC-01_7, SC-01_13, POL-01_63, SC-01_12]","[0.4939, 0.4637, 0.4496, 0.4196, 0.4184]",2.0,False,True,True
2,So my hard drive with my assignment stopped wo...,[SC-01_16],EXT-01_1,"[EXT-01_1, EXT-01_0, EAA-02_34]","[EXT-01_1, EXT-01_0, EAA-02_34, EAA-02_35, EAA...","[0.4814, 0.4718, 0.4334, 0.4215, 0.4156]",NaN,False,False,False
3,I've just booked my first appointment with ELS...,[EAA-02_29],EAA-02_29,"[EAA-02_29, EAA-01_18, EAA-02_20]","[EAA-02_29, EAA-01_18, EAA-02_20, EAA-02_21, E...","[0.3359, 0.3117, 0.2973, 0.2918, 0.2844]",1.0,True,True,True
4,I emailed my course coordinator three days ago...,[EAA-02_38],EAA-02_38,"[EAA-02_38, EXT-01_0, EAA-02_35]","[EAA-02_38, EXT-01_0, EAA-02_35, POL-01_62, EA...","[0.6137, 0.5667, 0.5502, 0.541, 0.5305]",1.0,True,True,True
5,Having a hard time reading through journal art...,[EAA-02_46],EAA-02_51,"[EAA-02_51, EAA-02_47, EAA-02_46]","[EAA-02_51, EAA-02_47, EAA-02_46, SC-01_15, EA...","[0.5002, 0.4879, 0.4547, 0.3887, 0.3754]",3.0,False,True,True
6,My mate got flagged for plagiarism because he ...,[POL-01_70],POL-01_70,"[POL-01_70, EXT-01_3, POL-01_56]","[POL-01_70, EXT-01_3, POL-01_56, POL-01_67, SC...","[0.517, 0.4462, 0.3762, 0.3481, 0.3135]",1.0,True,True,True
7,Who actually sets RMIT's official academic cal...,[POL-01_67],POL-01_67,"[POL-01_67, EAA-02_50, EAA-02_52]","[POL-01_67, EAA-02_50, EAA-02_52, POL-01_56, E...","[0.4506, 0.3786, 0.3535, 0.3338, 0.3252]",1.0,True,True,True
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]",EAA-01_18,"[EAA-01_18, POL-01_63, SC-01_8]","[EAA-01_18, POL-01_63, SC-01_8, EAA-02_43, POL...","[0.5787, 0.5757, 0.5519, 0.541, 0.514]",3.0,False,True,True
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EXT-01_0,"[EXT-01_0, EAA-02_41, EAA-02_34]","[EXT-01_0, EAA-02_41, EAA-02_34, EXT-01_1, POL...","[0.5287, 0.5211, 0.5024, 0.4996, 0.4908]",4.0,False,False,True


In [7]:
# Overall retrieval performance
top1_hit_rate = evaluation_df["top1_hit"].mean()
top3_hit_rate = evaluation_df["top3_hit"].mean()
top5_hit_rate = evaluation_df["top5_hit"].mean()

print("Retrieval Evaluation")
print("--------------------")
print(f"Top-1 Hit Rate: {top1_hit_rate:.1%}")
print(f"Top-3 Hit Rate: {top3_hit_rate:.1%}")
print(f"Top-5 Hit Rate: {top5_hit_rate:.1%}")

Retrieval Evaluation
--------------------
Top-1 Hit Rate: 50.0%
Top-3 Hit Rate: 75.0%
Top-5 Hit Rate: 91.7%


In [8]:
# Compact results table for reporting
summary_df = evaluation_df[
    [
        "prompt",
        "expected_chunks",
        "top1_chunk",
        "top3_chunks",
        "top5_chunks",
        "expected_rank_in_top5",
        "top1_hit",
        "top3_hit",
        "top5_hit"
    ]
].copy()

display(summary_df)

,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5,top1_hit,top3_hit,top5_hit
0,I've got a timed quiz that I won't be able to ...,[EXT-01_1],EXT-01_1,"[EXT-01_1, EXT-01_0, POL-01_62]","[EXT-01_1, EXT-01_0, POL-01_62, EXT-01_2, EAA-...",1.0,True,True,True
1,"I've recently lost my job, is this a valid rea...",[SC-01_7],SC-01_4,"[SC-01_4, SC-01_7, SC-01_13]","[SC-01_4, SC-01_7, SC-01_13, POL-01_63, SC-01_12]",2.0,False,True,True
2,So my hard drive with my assignment stopped wo...,[SC-01_16],EXT-01_1,"[EXT-01_1, EXT-01_0, EAA-02_34]","[EXT-01_1, EXT-01_0, EAA-02_34, EAA-02_35, EAA...",NaN,False,False,False
3,I've just booked my first appointment with ELS...,[EAA-02_29],EAA-02_29,"[EAA-02_29, EAA-01_18, EAA-02_20]","[EAA-02_29, EAA-01_18, EAA-02_20, EAA-02_21, E...",1.0,True,True,True
4,I emailed my course coordinator three days ago...,[EAA-02_38],EAA-02_38,"[EAA-02_38, EXT-01_0, EAA-02_35]","[EAA-02_38, EXT-01_0, EAA-02_35, POL-01_62, EA...",1.0,True,True,True
5,Having a hard time reading through journal art...,[EAA-02_46],EAA-02_51,"[EAA-02_51, EAA-02_47, EAA-02_46]","[EAA-02_51, EAA-02_47, EAA-02_46, SC-01_15, EA...",3.0,False,True,True
6,My mate got flagged for plagiarism because he ...,[POL-01_70],POL-01_70,"[POL-01_70, EXT-01_3, POL-01_56]","[POL-01_70, EXT-01_3, POL-01_56, POL-01_67, SC...",1.0,True,True,True
7,Who actually sets RMIT's official academic cal...,[POL-01_67],POL-01_67,"[POL-01_67, EAA-02_50, EAA-02_52]","[POL-01_67, EAA-02_50, EAA-02_52, POL-01_56, E...",1.0,True,True,True
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]",EAA-01_18,"[EAA-01_18, POL-01_63, SC-01_8]","[EAA-01_18, POL-01_63, SC-01_8, EAA-02_43, POL...",3.0,False,True,True
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EXT-01_0,"[EXT-01_0, EAA-02_41, EAA-02_34]","[EXT-01_0, EAA-02_41, EAA-02_34, EXT-01_1, POL...",4.0,False,False,True


In [9]:
# Show failed Top-1 cases for error analysis
top1_failures = evaluation_df[
    evaluation_df["top1_hit"] == False
][
    [
        "prompt",
        "expected_chunks",
        "top1_chunk",
        "top3_chunks",
        "top5_chunks",
        "expected_rank_in_top5"
    ]
]

print("Top-1 failures:", len(top1_failures))
display(top1_failures)

Top-1 failures: 6


,prompt,expected_chunks,top1_chunk,top3_chunks,top5_chunks,expected_rank_in_top5
1,"I've recently lost my job, is this a valid rea...",[SC-01_7],SC-01_4,"[SC-01_4, SC-01_7, SC-01_13]","[SC-01_4, SC-01_7, SC-01_13, POL-01_63, SC-01_12]",2.0
2,So my hard drive with my assignment stopped wo...,[SC-01_16],EXT-01_1,"[EXT-01_1, EXT-01_0, EAA-02_34]","[EXT-01_1, EXT-01_0, EAA-02_34, EAA-02_35, EAA...",NaN
5,Having a hard time reading through journal art...,[EAA-02_46],EAA-02_51,"[EAA-02_51, EAA-02_47, EAA-02_46]","[EAA-02_51, EAA-02_47, EAA-02_46, SC-01_15, EA...",3.0
8,"I missed a test because I was unwell, How long...","[SC-01_8, EAA-02_43]",EAA-01_18,"[EAA-01_18, POL-01_63, SC-01_8]","[EAA-01_18, POL-01_63, SC-01_8, EAA-02_43, POL...",3.0
9,My nan just passed away but I've got a group a...,"[POL-01_62, EXT-01_1]",EXT-01_0,"[EXT-01_0, EAA-02_41, EAA-02_34]","[EXT-01_0, EAA-02_41, EAA-02_34, EXT-01_1, POL...",4.0
10,I think my tutor marked me down assuming I pla...,"[POL-01_58, POL-01_59]",EXT-01_3,"[EXT-01_3, POL-01_70, EAA-02_38]","[EXT-01_3, POL-01_70, EAA-02_38, POL-01_59, EX...",4.0


In [10]:
# Save the detailed evaluation results
output_path = "./05_rag_test_evaluation_results.csv"

evaluation_df.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)

Saved: ./05_rag_test_evaluation_results.csv


In [12]:
# Save a simple metrics summary
metrics_df = pd.DataFrame({
    "metric": ["Top-1 Hit Rate", "Top-3 Hit Rate", "Top-5 Hit Rate"],
    "value": [top1_hit_rate, top3_hit_rate, top5_hit_rate]
})

display(metrics_df)

metrics_df.to_csv(
    "./05_rag_retrieval_metrics.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Metrics saved to ./rag_retrieval_metrics.csv")

,metric,value
0,Top-1 Hit Rate,0.500000
1,Top-3 Hit Rate,0.750000
2,Top-5 Hit Rate,0.916667


Metrics saved to ./rag_retrieval_metrics.csv


In [13]:
case_number = 0

row = evaluation_df.iloc[case_number]

print("Prompt:")
print(row["prompt"])
print("\nExpected chunk(s):", row["expected_chunks"])
print("\nTop-5 retrieved chunks:", row["top5_chunks"])
print("\nTop-5 scores:", row["top5_scores"])

print("\nRetrieved chunk details:")
for rank, chunk_id in enumerate(row["top5_chunks"], start=1):
    chunk = chunks_df[chunks_df["chunk_id"] == chunk_id].iloc[0]

    print("\n" + "-" * 60)
    print("Rank:", rank)
    print("Chunk ID:", chunk["chunk_id"])
    print("Source:", chunk["source_id"])
    print("Section:", chunk["section"])
    print("Content:", chunk["content"][:500])

Prompt:
I've got a timed quiz that I won't be able to complete on time, am I able to get an extension?

Expected chunk(s): ['EXT-01_1']

Top-5 retrieved chunks: ['EXT-01_1', 'EXT-01_0', 'POL-01_62', 'EXT-01_2', 'EAA-02_35']

Top-5 scores: [0.6777, 0.6126, 0.4892, 0.4775, 0.4761]

Retrieved chunk details:

------------------------------------------------------------
Rank: 1
Chunk ID: EXT-01_1
Source: EXT-01
Section: Assessments eligible for an extension
Content: You can apply for an extension for assessments which have a deadline, such as assignments, essays and projects, but not for timed assessments such as tests, exams, quizzes or lab/practical assessments. If you want to apply for assessment flexibility for a timed assessment, you must apply for special consideration .

------------------------------------------------------------
Rank: 2
Chunk ID: EXT-01_0
Source: EXT-01
Section: If you can't submit an assessment on time due to unforeseen circumstances outside your control, you may 